# DS Week 03 — Causal Uplift Modeling for Incremental Marketing Targeting
Estimate heterogeneous treatment effects from a randomized email experiment and convert them into a budget-aware targeting policy. The notebook asks who converts *because of* treatment, not merely who is likely to convert.

## 1. Business problem, hypotheses, and decision
Hypotheses: average email effect is positive but heterogeneous; recent/high-history/multichannel customers may differ in responsiveness; causal ranking should outperform random targeting. Decision: rank by estimated CATE and target under budget constraints.

## 2. Dataset provenance, license/access, target
Primary source: Hillstrom MineThatData randomized email experiment (64k customers, three arms). Treatment is any email vs no email; target is conversion. See DATASET.md. If the real CSV is unavailable, execute deterministic synthetic smoke data; synthetic results are not Hillstrom benchmark results.

In [ ]:
from pathlib import Path
import sys,json,warnings,numpy as np,pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings('ignore')
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd();sys.path.insert(0,str(ROOT))
from src.models import SLearner,TLearner,XLearner
from src.metrics import ate,qini_auc,policy_value_ipw
SEED=42;rng=np.random.default_rng(SEED)

## 3. Load data / deterministic fallback

In [ ]:
real_path=ROOT/'dataset/hillstrom.csv'
if real_path.exists():
 raw=pd.read_csv(real_path);raw.columns=[c.strip().lower().replace(' ','_') for c in raw.columns];df=raw.copy();df['treatment']=(df['segment'].str.lower()!='no_e-mail').astype('int8');source_mode='REAL_HILLSTROM'
else:
 n=12000;recency=rng.integers(1,13,n);history=np.exp(rng.normal(5,.8,n)).clip(10,4000);mens=rng.binomial(1,.45,n);womens=rng.binomial(1,.48,n);newbie=rng.binomial(1,.28,n);channel=rng.choice(['Phone','Web','Multichannel'],n,p=[.35,.40,.25]);zip_code=rng.choice(['Urban','Suburban','Rural'],n,p=[.42,.38,.20]);treatment=rng.binomial(1,2/3,n);base=-4.4+.00025*history-.055*recency+.18*(channel=='Multichannel')+.15*newbie;tau=.35+.30*(recency<=4)+.28*(channel=='Multichannel')-.20*(history>1200)+.18*mens;prob=1/(1+np.exp(-(base+treatment*tau)));conversion=rng.binomial(1,prob);df=pd.DataFrame(dict(recency=recency,history=history,mens=mens,womens=womens,newbie=newbie,channel=channel,zip_code=zip_code,treatment=treatment,conversion=conversion));source_mode='SYNTHETIC_SMOKE'
print(source_mode,df.shape)

## 4. Leakage analysis
Only pre-treatment variables are features. Exclude conversion, visit, spend, segment, and treatment from the normal feature matrix. Visit/spend are post-treatment outcomes; segment only constructs treatment.

In [ ]:
post={'conversion','visit','spend','segment','treatment'};candidate=[c for c in df.columns if c not in post];assert not ({'visit','spend','conversion','segment'}&set(candidate));candidate

## 5. EDA, missingness, outliers, treatment balance

In [ ]:
print(df.isna().mean().sort_values(ascending=False).head());print(df.groupby('treatment')['conversion'].agg(['count','mean']));print('ATE',ate(df.conversion.values,df.treatment.values))
balance=[]
for c in [x for x in ['recency','history','mens','womens','newbie'] if x in df]:
 a=df.loc[df.treatment==1,c].astype(float);b=df.loc[df.treatment==0,c].astype(float);balance.append((c,(a.mean()-b.mean())/(np.sqrt((a.var()+b.var())/2)+1e-9)))
pd.DataFrame(balance,columns=['feature','SMD'])

## 6. Domain-aware feature engineering
Leakage-safe pre-treatment features: log historical spend, history per active month, recent-customer flag, cross-category prior purchase and multichannel proxy.

In [ ]:
d=df.copy();d['log_history']=np.log1p(d['history']);d['history_per_active_month']=d['history']/(13-d['recency'].clip(1,12));d['recent_customer']=(d['recency']<=4).astype('int8');d['cross_category_buyer']=((d['mens']==1)&(d['womens']==1)).astype('int8');d['channel_breadth_proxy']=(d['channel'].astype(str).str.lower()=='multichannel').astype('int8')
features=[c for c in d.columns if c not in {'conversion','visit','spend','segment','treatment'}];cat=[c for c in features if d[c].dtype=='object'];num=[c for c in features if c not in cat];pre=ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cat)]);X=pre.fit_transform(d[features]).astype('float32');y=d.conversion.to_numpy('int8');w=d.treatment.to_numpy('int8');print(X.shape,X.nbytes/1e6)

## 7. Train/validation/test strategy
Customer-level randomized assignment permits stratified random splitting. The test set stays untouched until final evaluation. Repeated-customer or temporal deployments require group-aware/forward splits.

In [ ]:
idx=np.arange(len(d));strata=w*2+y;train_idx,temp_idx=train_test_split(idx,test_size=.4,random_state=SEED,stratify=strata);val_idx,test_idx=train_test_split(temp_idx,test_size=.5,random_state=SEED,stratify=strata[temp_idx]);print(len(train_idx),len(val_idx),len(test_idx))

## 8. Baselines and competitive models
Compare constant ATE, ordinary response ranking, S-learner, T-learner and X-style learner. Response ranking is included to demonstrate that outcome propensity is not treatment effect.

In [ ]:
response=LogisticRegression(max_iter=500,class_weight='balanced').fit(X[train_idx],y[train_idx]);response_score=response.predict_proba(X[val_idx])[:,1];constant=np.repeat(ate(y[train_idx],w[train_idx]),len(val_idx));models={'S':SLearner().fit(X[train_idx],w[train_idx],y[train_idx]),'T':TLearner().fit(X[train_idx],w[train_idx],y[train_idx]),'X':XLearner().fit(X[train_idx],w[train_idx],y[train_idx])};val_scores={k:m.effect(X[val_idx]) for k,m in models.items()};rows=[('constant_ATE',qini_auc(y[val_idx],w[val_idx],constant)),('response_model',qini_auc(y[val_idx],w[val_idx],response_score))]+[(k,qini_auc(y[val_idx],w[val_idx],v)) for k,v in val_scores.items()];pd.DataFrame(rows,columns=['model','validation_qini']).sort_values('validation_qini',ascending=False)

## 9. Hyperparameter strategy
Use constrained tree complexity and select on validation Qini rather than AUROC. A production run should use a small depth/learning-rate/regularization search, not an expensive grid.

In [ ]:
best_name=max(val_scores,key=lambda k:qini_auc(y[val_idx],w[val_idx],val_scores[k]));best=models[best_name];test_uplift=best.effect(X[test_idx]);print(best_name,qini_auc(y[test_idx],w[test_idx],test_uplift))

## 10. Evaluation metrics tied to business costs
Primary metric is Qini/incremental ranking. Held-out inverse-propensity policy value evaluates the treatment policy under randomized assignment.

In [ ]:
rows=[]
for frac in [.1,.2,.3,.5,1.0]:
 threshold=np.quantile(test_uplift,1-frac) if frac<1 else -np.inf;policy=(test_uplift>=threshold).astype(int);rows.append((frac,policy.mean(),policy_value_ipw(y[test_idx],w[test_idx],policy,p=float(w[train_idx].mean()))))
policy_df=pd.DataFrame(rows,columns=['budget_fraction','targeted_fraction','ipw_policy_value']);policy_df

## 11. Error/slice analysis
CATE is not individually observable, so inspect empirical treatment-control differences within model-ranked deciles and business-relevant cohorts rather than claiming individual ground truth.

In [ ]:
test=d.iloc[test_idx].copy();test['uplift_score']=test_uplift;test['uplift_decile']=pd.qcut(test.uplift_score,10,labels=False,duplicates='drop');[(int(k),len(g),ate(g.conversion.values,g.treatment.values)) for k,g in test.groupby('uplift_decile') if g.treatment.nunique()==2]

## 12. Explainability / interpretability
Feature associations do not prove heterogeneous causal mechanisms. Inspect uplift stability across pre-treatment business slices and treat causal interpretation conservatively.

In [ ]:
test.groupby('recent_customer').apply(lambda g:pd.Series({'n':len(g),'mean_pred_uplift':g.uplift_score.mean(),'empirical_ate':ate(g.conversion.values,g.treatment.values) if g.treatment.nunique()==2 else np.nan}))

## 13. Calibration / uncertainty: bootstrap Qini
Bootstrap the held-out test set to avoid over-interpreting a single uplift-ranking estimate.

In [ ]:
r=np.random.default_rng(SEED+1);boot=[]
for _ in range(100):
 b=r.integers(0,len(test_idx),len(test_idx));boot.append(qini_auc(y[test_idx][b],w[test_idx][b],test_uplift[b]))
lo,med,hi=np.percentile(boot,[2.5,50,97.5]);print({'median':med,'qini_95pct':[lo,hi]})

## 14. Robustness / sensitivity checks
Check targeting budgets, customer-history slices, treatment-rate stability and bootstrap variation. Randomization reduces confounding concerns but not variance, overlap or external-validity concerns.

In [ ]:
print('Treatment rate train/val/test',w[train_idx].mean(),w[val_idx].mean(),w[test_idx].mean());print(policy_df)

## 15. Experiment comparison, conclusions, limitations, production/research follow-ups
Promote only if causal ranking beats constant/random targeting on held-out incremental metrics and remains stable. Limitations: individual effects are unobserved; combining email arms discards multi-treatment structure; one historical campaign may not transport; campaign economics are simplified; synthetic fallback is not a Hillstrom benchmark. Follow-ups: full Hillstrom multi-arm analysis, DR/R-learners, causal forests, cross-fitting, honest policy-value intervals, drift monitoring and prospective randomized policy tests.

## 16. Reproducibility and hardware
Seed=42. Full Hillstrom (~64k rows) is CPU-friendly on Ryzen 7/16GB. Dense matrices use float32; GPU is unnecessary. Criteo-scale work should use columnar/chunked ingestion, sparse matrices, histogram learners and distributed evaluation.

In [ ]:
summary={'source_mode':source_mode,'n_rows':int(len(d)),'selected_model':best_name,'test_qini':float(qini_auc(y[test_idx],w[test_idx],test_uplift)),'bootstrap_qini_95':[float(lo),float(hi)],'test_ate':float(ate(y[test_idx],w[test_idx]))};(ROOT/'artifacts/metrics.json').write_text(json.dumps(summary,indent=2));summary